# LX · 00 · Build the D3 LCMV / Xu-protocol dataset
OpenNeuro ds004504 v1.0.9 derivatives (**already on Drive, not downloaded again**): 36 AD + 29 HC (FTD excluded by
design), 19 channels with the verified EEGLAB electrode coordinates. Every EEGLAB `boundary` cuts the recording;
each continuous segment is resampled on its own; five minutes of usable EEG per participant are selected
deterministically; 10 s windows with a 1 s step (90 % overlap) never cross a boundary or its filter margin.
Per participant: LCMV beamformer (own data covariance, rank-aware, unit-noise-gain), 68 Desikan-Killiany ROIs,
five Xu bands (delta 0.5-4, theta 4-8, alpha 8-13, beta 13-25, gamma 25-45 Hz), PSD, differential entropy,
PLV and |ciPLV|. Two graph stores: **mode B** (all accepted windows) and **mode A** (Xu tenfold: 10 samples per
participant, 65 -> 650).

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** D3 LCMV beamforming + Xu et al. (2025) protocol (methodology log entry 50, `docs/lcmv_xu.md`). All outputs
are under `<outputs>/D3_LCMV_XU/`; D1/D2 and the earlier D3 (wMNE) runs and results are never touched.
**Steps:** 1. input check (participants, channels, coordinates) · 2. recording QC of all 65 participants (durations,
boundaries, segments, five-minute selection; no source model) · 3. smoke test: one participant through the full
LCMV path, verified against MNE's own `apply_lcmv_raw` · 4. full build (starts automatically only if the smoke test
passed; resumable per participant) · 5. validation (no window crosses a boundary, frozen participant split,
shapes, finiteness) · 6. participant QC report. A GPU is not needed for this notebook.

In [ ]:
# ===== D3 LCMV / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_LX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The LCMV / Xu suite has its own code pin, <outputs>/D3_LCMV_XU/CODE_COMMIT.txt (all LX notebooks run
# that commit); the D1/D2 and D3 pins are only reported, never changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'LX']
     + (['--update-code'] if UPDATE_LX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
print('suite outputs:', LX)

## 1. Input check (36 AD / 29 HC, 19 channels and verified electrode coordinates in every file)

In [ ]:
sh("python scripts/d3_data.py check")

## 2. Recording QC of every participant (no source model)

In [ ]:
sh("python scripts/lx_build.py inspect")

## 3. Smoke test: one participant, full LCMV path (separate `_limit1` run)

In [ ]:
sh("python scripts/lx_build.py build --limit 1")
sh("python scripts/lx_build.py validate --limit 1")
SMOKE_PASSED = True                       # reached only if both commands above succeeded

## 4. Full build (all 65 participants; resumable, completed participants are skipped)

In [ ]:
if not SMOKE_PASSED:
    raise RuntimeError('smoke test did not pass; the full build is not started')
sh("python scripts/lx_build.py build")

## 5. Validation of both graph stores

In [ ]:
sh("python scripts/lx_build.py validate")

## 6. Participant QC report

In [ ]:
import glob, json
import pandas as pd
run = sorted(p for p in glob.glob(LX + '/graphsLX_v*') if '_limit' not in p)[-1]
qc = pd.read_csv(run + '/qc_report.csv')
pd.set_option('display.max_rows', 100, 'display.width', 250)
display(qc[[c for c in ['key', 'label', 'original_duration_s', 'n_boundary_events', 'n_segments', 'n_usable_segments',
                        'selection_mode', 'selected_duration_s', 'n_windows_candidate', 'n_windows_accepted',
                        'n_windows_rejected', 'n_tenfold', 'lcmv_rank', 'status', 'reason'] if c in qc]])
print(qc['status'].value_counts().to_dict(), '| windows (mode B):', int(qc['n_windows_kept'].sum()),
      '| samples (mode A):', int(qc['n_tenfold'].sum()))
print('frozen split:', {k: v for k, v in json.load(open(run + '/xu_split.json')).items() if k.startswith('n_')})